# Amazon Search Ranking — Week 8 V4 Final Package

**Upload only three files:**
1. `amazon_week8_v1_final_benchmark_results.zip`
2. `amazon_week8_v2_ablation_progression_results.zip`
3. `amazon_week8_v3_error_analysis_results.zip`

The notebook creates the final consolidated report package, tables, figures, presentation outline, demo script, and README.

In [ ]:

# ============================================================
# AMAZON SEARCH RANKING — WEEK 8 V4 FINAL PACKAGE
#
# Upload ONLY:
# 1) amazon_week8_v1_final_benchmark_results.zip
# 2) amazon_week8_v2_ablation_progression_results.zip
# 3) amazon_week8_v3_error_analysis_results.zip
#
# Output:
# amazon_week8_v4_final_package.zip
# ============================================================

import os, zipfile, shutil, glob
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from google.colab import files
    uploaded = files.upload()
    paths = [os.path.abspath(n) for n in uploaded.keys()]
except Exception:
    paths = glob.glob("*.zip")

def pick(pattern):
    hits = [p for p in paths if pattern.lower() in os.path.basename(p).lower()]
    if not hits:
        raise FileNotFoundError(f"Missing required ZIP: {pattern}")
    return hits[0]

V1 = pick("week8_v1_final_benchmark_results")
V2 = pick("week8_v2_ablation_progression_results")
V3 = pick("week8_v3_error_analysis_results")

OUT = Path("/content/week8_v4_final_package")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()
(OUT/"figures").mkdir()
(OUT/"tables").mkdir()
(OUT/"docs").mkdir()

def read_csv_zip(zp, member):
    with zipfile.ZipFile(zp) as z:
        with z.open(member) as f:
            return pd.read_csv(f)

def copy_member(zp, member, dst):
    with zipfile.ZipFile(zp) as z:
        with z.open(member) as src, open(dst,"wb") as f:
            shutil.copyfileobj(src,f)

ranking = read_csv_zip(V1,"week8_v1_ranking_benchmark.csv")
retrieval = read_csv_zip(V1,"week8_v1_retrieval_benchmark.csv")
cold = read_csv_zip(V1,"week8_v1_coldstart_benchmark.csv")
system = read_csv_zip(V1,"week8_v1_system_metrics.csv")
latency = read_csv_zip(V1,"week8_v1_stage_latency.csv")
progression = read_csv_zip(V2,"week8_v2_model_progression.csv")
ablation = read_csv_zip(V2,"week8_v2_ablation_contribution.csv")
err_cat = read_csv_zip(V3,"week8_v3_error_category_summary.csv")
err_sev = read_csv_zip(V3,"week8_v3_error_severity_summary.csv")
rerank_cases = read_csv_zip(V3,"week8_v3_representative_reranking_failures.csv")
retr_cases = read_csv_zip(V3,"week8_v3_representative_retrieval_failures.csv")
retr_err_summary = read_csv_zip(V3,"week8_v3_retrieval_error_summary.csv")

tables = {
    "final_ranking_benchmark.csv":ranking,
    "full_catalog_retrieval_benchmark.csv":retrieval,
    "coldstart_benchmark.csv":cold,
    "model_progression.csv":progression,
    "feature_ablation_contribution.csv":ablation,
    "reranking_error_summary.csv":err_cat,
    "error_severity_summary.csv":err_sev,
    "representative_reranking_failures.csv":rerank_cases,
    "representative_retrieval_failures.csv":retr_cases,
    "retrieval_error_summary.csv":retr_err_summary,
    "stage_latency.csv":latency,
    "system_metrics.csv":system
}
for fn,df in tables.items():
    df.to_csv(OUT/"tables"/fn,index=False)

figs = [
    (V1,"week8_v1_ranking_ndcg10.png","01_final_ranking_ndcg10.png"),
    (V2,"week8_v2_model_progression.png","02_model_progression.png"),
    (V2,"week8_v2_ablation_contribution.png","03_feature_ablation.png"),
    (V2,"week8_v2_coldstart_comparison.png","04_coldstart_comparison.png"),
    (V2,"week8_v2_retrieval_tradeoff.png","05_retrieval_tradeoff.png"),
    (V3,"week8_v3_error_category_counts.png","06_error_categories.png"),
    (V3,"week8_v3_worst15_reranking_queries.png","07_worst15_queries.png"),
    (V3,"week8_v3_retrieval_failure_map.png","08_retrieval_failure_map.png"),
    (V1,"week8_v1_stage_latency.png","09_stage_latency.png"),
    (V1,"week8_v1_system_architecture.png","10_system_architecture.png")
]
for zp,member,name in figs:
    copy_member(zp,member,OUT/"figures"/name)

best = ranking.sort_values("NDCG@10",ascending=False).iloc[0]
bm25 = float(ranking.loc[ranking["model"].eq("BM25"),"NDCG@10"].iloc[0])
cold_lm = float(cold.loc[cold["model"].eq("LambdaMART"),"NDCG@10"].iloc[0])
cold_zs = float(cold.loc[cold["model"].eq("ZeroShot+LLM"),"NDCG@10"].iloc[0])
fresh = float(system.loc[system["metric"].eq("V1.3 fresh mean latency (ms)"),"value"].iloc[0])
n_fail = int(err_cat["query_count"].sum())

summary=f"""# Week 8 Final Executive Summary

- Best ranking model: {best['model']}
- NDCG@10: {best['NDCG@10']:.4f}
- Relative gain vs BM25: {(best['NDCG@10']/bm25-1)*100:.2f}%
- MRR: {best['MRR']:.4f}
- ZeroShot+LLM cold-start NDCG@10: {cold_zs:.4f}
- LambdaMART cold-start NDCG@10: {cold_lm:.4f}
- Reranking queries below 0.90 NDCG@10: {n_fail}/200
- Fresh serving mean latency: {fresh:.2f} ms

Final conclusion:
The best system combines lexical, semantic, multimodal and structural evidence with LambdaMART,
plus a zero-shot/LLM cold-start fallback and optimized serving pipeline.
"""
(OUT/"docs"/"week8_final_executive_summary.md").write_text(summary,encoding="utf-8")

# reuse the detailed report-ready analyses from V2 and V3
with zipfile.ZipFile(V2) as z:
    v2txt = z.read("week8_v2_report_ready_analysis.md").decode("utf-8")
with zipfile.ZipFile(V3) as z:
    v3txt = z.read("week8_v3_report_ready_error_analysis.md").decode("utf-8")

technical=f"""# Amazon Product Search Ranking System — Final Technical Report

## Executive Summary
LambdaMART achieved NDCG@10={best['NDCG@10']:.4f}, a {(best['NDCG@10']/bm25-1)*100:.2f}% relative improvement over BM25.
Fresh end-to-end latency was {fresh:.2f} ms.

## Project Roadmap
Week 1 preprocessing; Week 2 lexical baselines; Week 3 Llama 3 query understanding;
Week 4 SBERT/CLIP retrieval; Week 5 learning-to-rank; Week 6 cold start;
Week 7 deployment; Week 8 benchmark, ablation and error analysis.

## Final Benchmark
See `tables/final_ranking_benchmark.csv`.

## Ablation and Model Progression
{v2txt}

## Error Analysis
{v3txt}

## Final Conclusion
Strong e-commerce search quality came from combining complementary signals and evaluating retrieval,
reranking, cold-start and serving performance separately.
"""
(OUT/"docs"/"week8_final_technical_report.md").write_text(technical,encoding="utf-8")

slides=f"""# 15-Minute Final Presentation

1. Title
2. Problem and goal
3. 8-week roadmap
4. Architecture — `figures/10_system_architecture.png`
5. Model progression — `figures/02_model_progression.png`
6. Final benchmark — `figures/01_final_ranking_ndcg10.png`
7. Ablation — `figures/03_feature_ablation.png`
8. Retrieval trade-off — `figures/05_retrieval_tradeoff.png`
9. Cold start — `figures/04_coldstart_comparison.png`
10. Error analysis — `figures/06_error_categories.png`
11. Failure examples
12. Latency — `figures/09_stage_latency.png`
13. Live demo
14. Limitations / next steps
15. Conclusion: LambdaMART {best['NDCG@10']:.4f} NDCG@10, +{(best['NDCG@10']/bm25-1)*100:.2f}% vs BM25, {fresh:.2f} ms latency
"""
(OUT/"docs"/"week8_final_presentation_outline.md").write_text(slides,encoding="utf-8")

demo="""# Demo Script
1. Introduce end-to-end Amazon product search.
2. Enter a representative query.
3. Explain candidate retrieval.
4. Explain LambdaMART reranking using lexical, SBERT, CLIP and metadata signals.
5. Show top results.
6. Run a harder subjective query.
7. Mention ZeroShot+LLM cold-start fallback.
8. Close with <200 ms optimized latency.
"""
(OUT/"docs"/"week8_demo_script.md").write_text(demo,encoding="utf-8")

readme=f"""# Amazon Product Search Ranking System
Final LambdaMART NDCG@10: {best['NDCG@10']:.4f}
Relative improvement over BM25: +{(best['NDCG@10']/bm25-1)*100:.2f}%
Fresh serving latency: {fresh:.2f} ms

Folders:
- docs/
- figures/
- tables/
"""
(OUT/"README.md").write_text(readme,encoding="utf-8")

checklist="""# Submission Checklist
- [x] Benchmark
- [x] Ablation
- [x] Error analysis
- [x] Architecture
- [x] Latency
- [x] Technical report draft
- [x] Presentation outline
- [x] Demo script
- [ ] Export report PDF
- [ ] Build presentation deck
- [ ] Record demo video
- [ ] Push final GitHub repository
"""
(OUT/"docs"/"week8_submission_checklist.md").write_text(checklist,encoding="utf-8")

pd.DataFrame([
    {"path":str(p.relative_to(OUT)),"size_bytes":p.stat().st_size}
    for p in sorted(OUT.rglob("*")) if p.is_file()
]).to_csv(OUT/"manifest.csv",index=False)

ZIP="/content/amazon_week8_v4_final_package.zip"
if os.path.exists(ZIP):
    os.remove(ZIP)
with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p,arcname=str(p.relative_to(OUT)))

print("=== WEEK 8 V4 COMPLETE ===")
print("Final model:",best["model"])
print("NDCG@10:",round(float(best["NDCG@10"]),4))
print("Gain vs BM25:",round((float(best["NDCG@10"])/bm25-1)*100,2),"%")
print("Cold-start ZeroShot+LLM:",round(cold_zs,4))
print("Fresh mean latency:",round(fresh,2),"ms")
print("Output:",ZIP)

try:
    files.download(ZIP)
except Exception:
    pass
